# Appliance Energy Forecasting — Part 1: Data Preparation & EDA

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_ROOT = "/content/drive/MyDrive/appliance-energy-forecasting"
os.makedirs(PROJECT_ROOT, exist_ok=True)
os.chdir(PROJECT_ROOT)


In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.rcParams["figure.figsize"] = (14, 5)
np.random.seed(0)

os.makedirs("outputs/figures", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)


In [ ]:
# load raw 10-minute data from UCI repository
url = "https://archive.ics.uci.edu/ml/machine-learning-databases/00374/energydata_complete.csv"
df = pd.read_csv(url)

# parse timestamp and set as index
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date").sort_index()

print(df.shape)
df.head()


In [ ]:
# check for missing values in each column
print(df.isna().sum().sort_values(ascending=False).head(10))

# check for missing timestamps (gaps in the 10-minute sequence)
full_range = pd.date_range(df.index.min(), df.index.max(), freq="10min")
missing_timestamps = full_range.difference(df.index)
print(f"expected {len(full_range)} timestamps, got {len(df)}")
print(f"missing timestamps: {len(missing_timestamps)}")


In [ ]:
# resample 10-minute data to hourly means
hourly = df.resample("h").mean()

# interpolate small gaps rather than dropping them
hourly = hourly.interpolate("time")
hourly = hourly.dropna()

print("hourly data shape:", hourly.shape)
hourly[["Appliances"]].head()


In [ ]:
# plot full series
fig, ax = plt.subplots()
hourly["Appliances"].plot(ax=ax, linewidth=0.8)
ax.set_title("Hourly Appliance Energy Use - Full Series")
ax.set_ylabel("Appliances (Wh)")
ax.set_xlabel("Date")
plt.tight_layout()
plt.savefig("outputs/figures/01_full_series.png", dpi=150)
plt.show()

# plot first two weeks to inspect daily pattern
fig, ax = plt.subplots()
hourly["Appliances"].iloc[:24*14].plot(ax=ax)
ax.set_title("Hourly Appliance Energy Use - First 2 Weeks")
ax.set_ylabel("Appliances (Wh)")
plt.tight_layout()
plt.savefig("outputs/figures/02_two_week_zoom.png", dpi=150)
plt.show()


In [ ]:
# distribution of target variable
fig, ax = plt.subplots()
hourly["Appliances"].hist(bins=50, ax=ax)
ax.set_title("Distribution of Hourly Appliance Energy Use")
ax.set_xlabel("Appliances (Wh)")
plt.tight_layout()
plt.savefig("outputs/figures/03_target_distribution.png", dpi=150)
plt.show()

print(hourly["Appliances"].describe())


In [ ]:
# seasonal decomposition, period=24 tests for daily seasonality in hourly data
decomposition = seasonal_decompose(hourly["Appliances"], model="additive", period=24)

fig = decomposition.plot()
fig.set_size_inches(14, 8)
plt.tight_layout()
plt.savefig("outputs/figures/04_seasonal_decomposition.png", dpi=150)
plt.show()


In [ ]:
# ADF: null hypothesis = non-stationary. p < 0.05 -> reject null -> stationary
# KPSS: null hypothesis = stationary. p < 0.05 -> reject null -> non-stationary
def run_stationarity_tests(series, name="series"):
    adf_result = adfuller(series.dropna())
    kpss_result = kpss(series.dropna(), regression="c", nlags="auto")

    print(f"--- {name} ---")
    print(f"ADF statistic: {adf_result[0]:.4f}, p-value: {adf_result[1]:.4f}")
    print(f"  -> {'stationary' if adf_result[1] < 0.05 else 'non-stationary'}")
    print(f"KPSS statistic: {kpss_result[0]:.4f}, p-value: {kpss_result[1]:.4f}")
    print(f"  -> {'non-stationary' if kpss_result[1] < 0.05 else 'stationary'}")
    print()

run_stationarity_tests(hourly["Appliances"], "Appliances (level)")
run_stationarity_tests(hourly["Appliances"].diff(), "Appliances (1st difference)")
run_stationarity_tests(hourly["Appliances"].diff(24), "Appliances (seasonal diff, lag 24)")


In [ ]:
# ACF/PACF on first-differenced series to identify candidate SARIMAX orders
fig, axes = plt.subplots(2, 1, figsize=(14, 8))
plot_acf(hourly["Appliances"].diff().dropna(), lags=72, ax=axes[0])
axes[0].set_title("ACF - first-differenced series (72 lags = 3 days)")
plot_pacf(hourly["Appliances"].diff().dropna(), lags=72, ax=axes[1])
axes[1].set_title("PACF - first-differenced series")
plt.tight_layout()
plt.savefig("outputs/figures/05_acf_pacf.png", dpi=150)
plt.show()


In [ ]:
# save cleaned hourly dataset for use in later notebooks
hourly.to_csv("data/processed/appliance_hourly.csv")
print("saved to data/processed/appliance_hourly.csv")
